# 14 SmolVLA 端到端：训练、评估、视频与诊断

            这个 Notebook 把 SmolVLA 的完整学习路径放在一个文件里：先看已经复现成功的权重和视频，再检查数据，最后给出 smoke、长训、严格评估和日志追踪命令。

            运行方式建议：第一次教学演示只开 `RUN_SMOKE=1 RUN_EVAL=1`；完整复现实验再开 `RUN_LONG_TRAIN=1`。长训输出会真实写回 Notebook 单元格，而不是粘贴静态日志。


In [1]:

from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

try:
    from IPython.display import HTML, Image, Markdown, Video, display
except Exception:
    class Markdown(str):
        pass

    class HTML(str):
        pass

    def Image(filename=None, width=None, **kwargs):
        return f"[image] {filename}"

    def Video(filename=None, embed=False, width=None, **kwargs):
        return f"[video] {filename}"

    def display(obj):
        print(obj)


def find_topic_root():
    override = (
        os.environ.get("AMD_TOPIC_ROOT")
        or os.environ.get("NOTEBOOK_TOPIC_ROOT")
        or os.environ.get("TOPIC_ROOT")
    )
    roots = [Path(override).expanduser()] if override else []
    cwd = Path.cwd().resolve()
    roots.extend([cwd, *cwd.parents])

    candidates = []
    for root in roots:
        candidates.extend(
            [
                root,
                root / "16-专题组队学习" / "04-AMD-ROCm策略复刻专题",
                root / "04-AMD-ROCm策略复刻专题",
            ]
        )
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError(
        "找不到 AMD ROCm 专题目录。请从仓库根目录、专题目录启动 Jupyter，"
        "或设置 AMD_TOPIC_ROOT。"
    )


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", TOPIC_ROOT / "external" / "04mujoco复现ACT、Pi0、SmolVLA")
).expanduser()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", TOPIC_ROOT / "data")).expanduser()
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", TOPIC_ROOT / "checkpoints")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))

# The AMD teaching workflow should be runnable from local datasets/checkpoints.
# Avoid surprising network calls during class or when AUP/Radeon Cloud cannot
# reach Hugging Face.
os.environ.setdefault("HF_HUB_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("TRANSFORMERS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_DATASETS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_HOME", str(Path(os.environ.get("CACHE_ROOT", OUTPUT_ROOT / "cache")) / "huggingface"))
os.environ.setdefault("HF_DATASETS_CACHE", str(Path(os.environ["HF_HOME"]) / "datasets"))

def public_path(path):
    path = Path(path)
    replacements = [
        (TOPIC_ROOT, "$TOPIC_ROOT"),
        (PROJECT_ROOT, "$PROJECT_ROOT"),
        (DATA_ROOT, "$DATA_ROOT"),
        (MODEL_ROOT, "$MODEL_ROOT"),
        (OUTPUT_ROOT, "$OUTPUT_ROOT"),
    ]
    value = str(path)
    for root, label in sorted(replacements, key=lambda item: len(str(item[0])), reverse=True):
        root_value = str(root)
        if root_value and value.startswith(root_value):
            return label + value[len(root_value):]
    return value


print("TOPIC_ROOT =", public_path(TOPIC_ROOT))
print("PROJECT_ROOT =", public_path(PROJECT_ROOT))
print("DATA_ROOT =", public_path(DATA_ROOT))
print("MODEL_ROOT =", public_path(MODEL_ROOT))
print("OUTPUT_ROOT =", public_path(OUTPUT_ROOT))


TOPIC_ROOT = $TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $DATA_ROOT
MODEL_ROOT = $MODEL_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT


In [2]:

def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(public_path(x) if isinstance(x, (str, Path)) else str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


def show_json(path, max_chars=5000):
    path = Path(path)
    if not path.exists():
        print("文件不存在：", public_path(path))
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))
    return data


def show_video(filename, title):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        try:
            display(Video(filename=str(path), embed=True, width=960, html_attributes="controls muted"))
        except TypeError:
            display(Video(filename=str(path), embed=True, width=960))
    else:
        print("缺少视频素材：", public_path(path))


def show_image(filename, title, width=960):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        try:
            display(Image(filename=str(path), width=width))
        except TypeError:
            display(Image(filename=str(path)))
    else:
        print("缺少图片素材：", public_path(path))


def run_cmd_preview(command, cwd=None):
    shown = [public_path(x) if isinstance(x, (str, Path)) else x for x in command]
    print("$", shlex.join([str(x) for x in shown]))
    if cwd:
        print("cwd =", public_path(cwd))


def tail_log(log_path, lines=40):
    path = Path(log_path)
    if not path.exists():
        print("日志不存在：", public_path(path))
        return
    content = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_SMOKE = env_flag("RUN_SMOKE")
RUN_LONG_TRAIN = env_flag("RUN_LONG_TRAIN")
RUN_EVAL = env_flag("RUN_EVAL")
EVAL_SCRIPT = Path(os.environ.get("EVAL_SCRIPT", PROJECT_ROOT / "eval_policy_success.py"))


_XVFB_PROCESS = None


def ensure_xvfb_display():
    """Start a lightweight virtual display for headless MuJoCo evaluation."""
    global _XVFB_PROCESS
    if os.environ.get("DISPLAY"):
        print("DISPLAY =", os.environ["DISPLAY"])
        return None
    xvfb_bin = shutil.which("Xvfb")
    if not xvfb_bin:
        print("没有发现 Xvfb；如遇 GLFW DISPLAY 报错，请先安装 xvfb。")
        return None
    display_id = os.environ.get("NOTEBOOK_XVFB_DISPLAY", ":99")
    _XVFB_PROCESS = subprocess.Popen(
        [xvfb_bin, display_id, "-screen", "0", "1280x1024x24"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    os.environ["DISPLAY"] = display_id
    print("已启动 Notebook 内部 Xvfb：DISPLAY =", display_id)
    return _XVFB_PROCESS


def ensure_project_layout():
    required = [PROJECT_ROOT / "asset" / "example_scene_y2.xml", PROJECT_ROOT / "mujoco_env"]
    missing = [path for path in required if not path.exists()]
    if missing:
        print("当前 PROJECT_ROOT 还不是可运行工程，缺少：")
        for path in missing:
            print(" -", public_path(path))
        print("请先设置 PROJECT_ROOT，再运行训练或评估单元。")
        return False
    return True


def write_json_yaml(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import yaml
        text = yaml.safe_dump(payload, allow_unicode=True, sort_keys=False)
    except Exception:
        text = json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    path.write_text(text, encoding="utf-8")
    print("写出配置：", public_path(path))
    return path


def make_lerobot_train_config(policy_type, dataset_repo_id, dataset_root, output_dir, steps, batch_size, chunk_size, n_action_steps, seed=42):
    save_freq = int(os.environ.get(f"{policy_type.upper()}_SAVE_FREQ", os.environ.get("SAVE_FREQ", str(steps))))
    return {
        "dataset": {
            "repo_id": dataset_repo_id,
            "root": str(dataset_root),
            "use_imagenet_stats": True,
        },
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "output_dir": str(output_dir),
        "job_name": Path(output_dir).name,
        "batch_size": int(batch_size),
        "steps": int(steps),
        "save_freq": max(1, save_freq),
        "log_freq": 20,
        "num_workers": 4,
        "seed": int(seed),
        "resume": False,
        "eval_freq": -1,
        "save_checkpoint": True,
        "use_policy_training_preset": True,
        "wandb": {"enable": False, "disable_artifact": True},
    }


def train_lerobot_config_in_notebook(config_path, enabled=False, progress_name="train"):
    """Run LeRobot offline training directly inside the notebook kernel.

    The notebook cell owns dataset creation, policy creation, optimizer steps,
    checkpoint saving, tqdm progress, and metric JSONL writing.
    """
    config_path = Path(config_path)
    print("config =", public_path(config_path))
    if not enabled:
        print("未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。")
        return None
    if not ensure_project_layout():
        return None

    import time
    from contextlib import nullcontext

    import draccus
    import torch
    from torch.amp import GradScaler
    from tqdm.auto import tqdm

    from lerobot.common.datasets.factory import make_dataset
    from lerobot.common.datasets.sampler import EpisodeAwareSampler
    from lerobot.common.optim.factory import make_optimizer_and_scheduler
    from lerobot.common.policies.factory import make_policy
    from lerobot.common.policies.utils import get_device_from_parameters
    from lerobot.common.utils.random_utils import set_seed
    from lerobot.common.utils.train_utils import get_step_checkpoint_dir, save_checkpoint, update_last_checkpoint
    from lerobot.common.utils.utils import get_safe_torch_device
    from lerobot.configs.train import TrainPipelineConfig

    cfg = draccus.parse(TrainPipelineConfig, config_path=config_path, args=[])
    cfg.validate()
    if cfg.seed is not None:
        set_seed(cfg.seed)

    device = get_safe_torch_device(cfg.policy.device, log=True)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

    print("Creating dataset...")
    dataset = make_dataset(cfg)
    print("Creating policy...")
    pretrained_override = os.environ.get(f"{cfg.policy.type.upper()}_PRETRAINED_PATH_OVERRIDE") or os.environ.get("POLICY_PRETRAINED_PATH_OVERRIDE")
    if pretrained_override and not cfg.resume:
        cfg.policy.pretrained_path = str(Path(pretrained_override))
        print("pretrained override =", public_path(cfg.policy.pretrained_path))
    elif cfg.policy.type == "pi0" and not cfg.resume:
        cfg.policy.pretrained_path = "lerobot/pi0"
    elif cfg.policy.type == "smolvla" and not cfg.resume:
        smolvla_base_candidates = [
            os.environ.get("SMOLVLA_BASE_PATH"),
            os.environ.get("SMOLVLA_PRETRAINED_BASE_PATH"),
            str(MODEL_ROOT / "smolvla_base" / "pretrained_model"),
            str(MODEL_ROOT / "lerobot_smolvla_base_legacy"),
            str(MODEL_ROOT / "lerobot_smolvla_base"),
        ]
        local_smolvla_base = next((Path(p) for p in smolvla_base_candidates if p and Path(p).exists()), None)
        if local_smolvla_base is not None:
            cfg.policy.pretrained_path = str(local_smolvla_base)
            print("local smolvla base =", public_path(cfg.policy.pretrained_path))
        else:
            cfg.policy.pretrained_path = "lerobot/smolvla_base"
    policy = make_policy(cfg=cfg.policy, ds_meta=dataset.meta)

    # Compatibility for newer Transformers: PaliGemmaForConditionalGeneration may expose
    # language_model as GemmaModel directly, while this LeRobot Pi0 code expects
    # language_model.model.  Use a non-Module proxy so checkpoints/state_dict stay clean.
    if cfg.policy.type == "pi0":
        try:
            lm = policy.model.paligemma_with_expert.paligemma.language_model
            if not hasattr(lm, "model"):
                class _LanguageModelCoreProxy:
                    def __init__(self, core):
                        self._core = core

                    def __getattr__(self, name):
                        return getattr(self._core, name)

                object.__setattr__(lm, "model", _LanguageModelCoreProxy(lm))
                print("patched Pi0 PaliGemma language_model.model compatibility proxy")
        except Exception as exc:
            print(f"Pi0 PaliGemma compatibility patch skipped: {exc}")

    policy.to(device)
    policy.train()

    optimizer, lr_scheduler = make_optimizer_and_scheduler(cfg, policy)
    grad_scaler = GradScaler(device.type, enabled=cfg.policy.use_amp)

    def _dataset_column_values(name):
        hf_dataset = getattr(dataset, "hf_dataset", None)
        if hf_dataset is None or name not in getattr(hf_dataset, "column_names", []):
            return None
        values = hf_dataset[name]
        try:
            return list(values)
        except TypeError:
            return [values[i] for i in range(len(values))]

    def _task_name_map():
        meta = getattr(dataset, "meta", None)
        tasks = getattr(meta, "tasks", None)
        if tasks is None:
            return {}
        if isinstance(tasks, dict):
            return {int(k): str(v) for k, v in tasks.items()}
        try:
            return {int(k): str(v) for k, v in dict(tasks).items()}
        except Exception:
            return {}

    def _make_weighted_sampler(generator):
        mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE", "").strip().lower()
        if not mode or mode in {"0", "none", "off", "false"}:
            return None, {"mode": "none"}
        weights = torch.ones(len(dataset), dtype=torch.double)
        info = {"mode": mode, "num_frames": len(dataset)}

        if "blue" in mode:
            blue_weight = float(os.environ.get("NOTEBOOK_BLUE_WEIGHT", "2.0"))
            mask = [False] * len(dataset)
            task_indices = _dataset_column_values("task_index")
            task_names = _task_name_map()
            if task_indices is not None and task_names:
                for idx, task_index in enumerate(task_indices):
                    task_text = task_names.get(int(task_index), "").lower()
                    mask[idx] = ("blue" in task_text) or ("蓝" in task_text)
            else:
                for column in ["task", "language_instruction", "instruction"]:
                    values = _dataset_column_values(column)
                    if values is None:
                        continue
                    for idx, value in enumerate(values):
                        text = str(value).lower()
                        mask[idx] = ("blue" in text) or ("蓝" in text)
                    break
            blue_count = int(sum(mask))
            if blue_count == 0:
                print("警告：NOTEBOOK_FRAME_WEIGHT_MODE=blue 但没有识别到 blue/蓝 指令帧，采样退回均匀。")
            else:
                for idx, is_blue in enumerate(mask):
                    if is_blue:
                        weights[idx] *= blue_weight
            info.update({"blue_weight": blue_weight, "blue_frames": blue_count})

        weight_file = os.environ.get("NOTEBOOK_FRAME_WEIGHT_JSON")
        if weight_file:
            payload = json.loads(Path(weight_file).read_text(encoding="utf-8"))
            for key, value in payload.items():
                weights[int(key)] *= float(value)
            info.update({"weight_json": public_path(weight_file), "json_entries": len(payload)})

        if float(weights.sum()) <= 0:
            raise ValueError("采样权重总和为 0。")
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True,
            generator=generator,
        )
        info.update(
            {
                "weight_min": float(weights.min()),
                "weight_max": float(weights.max()),
                "weight_mean": float(weights.mean()),
            }
        )
        return sampler, info

    generator = torch.Generator()
    if cfg.seed is not None:
        generator.manual_seed(int(cfg.seed))

    weighted_sampler, sampler_info = _make_weighted_sampler(generator)
    if weighted_sampler is not None:
        shuffle = False
        sampler = weighted_sampler
        print("Notebook weighted sampler =", json.dumps(sampler_info, ensure_ascii=False))
    elif hasattr(cfg.policy, "drop_n_last_frames"):
        shuffle = False
        sampler = EpisodeAwareSampler(
            dataset.episode_data_index,
            drop_n_last_frames=cfg.policy.drop_n_last_frames,
            shuffle=True,
        )
    else:
        shuffle = True
        sampler = None

    dataloader = torch.utils.data.DataLoader(
        dataset,
        num_workers=cfg.num_workers,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        sampler=sampler,
        generator=generator if sampler is None else None,
        pin_memory=device.type != "cpu",
        drop_last=False,
    )
    # Do not use itertools.cycle here: it caches every batch and can exhaust
    # host RAM during a long Notebook training run. Recreate the iterator only
    # when the finite DataLoader is exhausted.
    dl_iter = iter(dataloader)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "notebook_train_metrics.jsonl"
    num_learnable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    num_total = sum(p.numel() for p in policy.parameters())
    print(f"output_dir = {public_path(output_dir)}")
    print(f"steps = {cfg.steps}, batch_size = {cfg.batch_size}, frames = {dataset.num_frames}, episodes = {dataset.num_episodes}")
    print(f"learnable_params = {num_learnable:,}, total_params = {num_total:,}")

    last_metrics = None
    progress = tqdm(range(1, cfg.steps + 1), desc=progress_name, dynamic_ncols=True)
    start_all = time.perf_counter()
    for step in progress:
        load_start = time.perf_counter()
        try:
            batch = next(dl_iter)
        except StopIteration:
            dl_iter = iter(dataloader)
            batch = next(dl_iter)
        data_s = time.perf_counter() - load_start
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                batch[key] = value.to(device, non_blocking=True)

        update_start = time.perf_counter()
        device_from_policy = get_device_from_parameters(policy)
        with torch.autocast(device_type=device_from_policy.type) if cfg.policy.use_amp else nullcontext():
            loss, output_dict = policy.forward(batch)
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy.parameters(),
            cfg.optimizer.grad_clip_norm,
            error_if_nonfinite=False,
        )
        grad_scaler.step(optimizer)
        grad_scaler.update()
        optimizer.zero_grad()
        if lr_scheduler is not None:
            lr_scheduler.step()
        if hasattr(policy, "update"):
            policy.update()
        update_s = time.perf_counter() - update_start

        is_log_step = cfg.log_freq > 0 and (step % cfg.log_freq == 0 or step == 1 or step == cfg.steps)
        is_saving_step = cfg.save_checkpoint and (step % cfg.save_freq == 0 or step == cfg.steps)
        if is_log_step:
            last_metrics = {
                "step": step,
                "loss": float(loss.detach().cpu()),
                "grad_norm": float(grad_norm.detach().cpu()) if hasattr(grad_norm, "detach") else float(grad_norm),
                "lr": float(optimizer.param_groups[0]["lr"]),
                "update_s": float(update_s),
                "data_s": float(data_s),
                "elapsed_s": float(time.perf_counter() - start_all),
            }
            with metrics_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(last_metrics, ensure_ascii=False) + "\n")
            progress.set_postfix(
                loss=f"{last_metrics['loss']:.4f}",
                lr=f"{last_metrics['lr']:.1e}",
                updt_s=f"{last_metrics['update_s']:.3f}",
            )
        if is_saving_step:
            checkpoint_dir = get_step_checkpoint_dir(cfg.output_dir, cfg.steps, step)
            print(f"\nSaving checkpoint at step {step}: {public_path(checkpoint_dir)}")
            save_checkpoint(checkpoint_dir, step, cfg, policy, optimizer, lr_scheduler)
            update_last_checkpoint(checkpoint_dir)

    print("训练完成。metrics =", public_path(metrics_path))
    if last_metrics is not None:
        print(json.dumps(last_metrics, ensure_ascii=False, indent=2))
    return {"output_dir": output_dir, "metrics_path": metrics_path, "last_metrics": last_metrics}


def load_eval_module():
    import importlib.util

    if not EVAL_SCRIPT.exists():
        raise FileNotFoundError(f"评估脚本不存在：{public_path(EVAL_SCRIPT)}")
    spec = importlib.util.spec_from_file_location("notebook_eval_policy_success", EVAL_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def run_eval_policy_in_notebook(
    policy_name,
    policy_path,
    result_path,
    episodes,
    seed_start,
    render=False,
    enabled=False,
    repo_id=None,
    dataset_root=None,
):
    print("policy =", policy_name)
    print("policy_path =", public_path(policy_path))
    print("result =", public_path(result_path))
    if not enabled:
        print("未启动。设置 RUN_EVAL=1 后，本单元会在 Notebook 内直接加载策略并闭环评估。")
        return None
    if not ensure_project_layout():
        return None

    import argparse
    from contextlib import contextmanager
    from tqdm.auto import tqdm

    @contextmanager
    def pushd(path):
        old = Path.cwd()
        os.chdir(path)
        try:
            yield
        finally:
            os.chdir(old)

    ensure_xvfb_display()
    module = load_eval_module()
    result_path = Path(result_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists():
        result_path.unlink()

    args = argparse.Namespace(
        policy=policy_name,
        episodes=int(episodes),
        seed_start=int(seed_start),
        max_action_steps=int(os.environ.get("EVAL_MAX_ACTION_STEPS", "400")),
        hz=float(os.environ.get("EVAL_HZ", "20")),
        render=bool(render),
        output_jsonl=result_path,
        device=os.environ.get("EVAL_DEVICE", "cuda"),
        reset_policy_each_action=env_flag("EVAL_RESET_POLICY_EACH_ACTION", False),
        act_n_action_steps=None,
        act_force_dataset_gripper=False,
        act_clamp_timestamp=False,
        act_policy_path=Path(policy_path),
        act_repo_id=repo_id or "datawhale_eai_pnp",
        act_dataset_root=Path(dataset_root or "./demo_data"),
        act_episode_timestamp_offsets="",
        act_episode_source_flags="",
        physical_success=env_flag("EVAL_PHYSICAL_SUCCESS", True),
        physical_min_lift=float(os.environ.get("EVAL_PHYSICAL_MIN_LIFT", "0.06")),
        physical_min_lift_steps=int(os.environ.get("EVAL_PHYSICAL_MIN_LIFT_STEPS", "3")),
        physical_final_upright_cos=float(os.environ.get("EVAL_PHYSICAL_FINAL_UPRIGHT_COS", "0.85")),
        smolvla_policy_path=Path(policy_path),
        pi0_policy_path=Path(policy_path),
        pi0_repo_id=repo_id or os.environ.get("PI0_DATASET_REPO_ID", "datawhale_eai_pnp_language"),
        pi0_dataset_root=Path(dataset_root or os.environ.get("PI0_DATASET_ROOT", "./demo_data_language")),
    )

    with pushd(PROJECT_ROOT):
        if policy_name == "act":
            policy = module.make_act_policy(
                args.device,
                args.act_policy_path,
                args.act_repo_id,
                args.act_dataset_root,
                n_action_steps=args.act_n_action_steps,
                episode_timestamp_offsets=args.act_episode_timestamp_offsets,
                episode_source_flags=args.act_episode_source_flags,
            )
            rollout = module.rollout_act
        elif policy_name == "smolvla":
            policy = module.make_smolvla_policy(args.device, args.smolvla_policy_path)
            rollout = module.rollout_language_policy
        elif policy_name == "pi0":
            policy = module.make_pi0_policy(args.device, args.pi0_policy_path, args.pi0_repo_id, args.pi0_dataset_root)
            rollout = module.rollout_language_policy
        else:
            raise ValueError(policy_name)

        rows = []
        for offset in tqdm(range(args.episodes), desc=f"{policy_name} eval", dynamic_ncols=True):
            seed = args.seed_start + offset
            row = rollout(args, policy, seed)
            rows.append(row)
            with result_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(json.dumps(row, ensure_ascii=False))
    summarize_jsonl(result_path)
    return rows


def list_checkpoints(run_dir):
    run_dir = Path(run_dir)
    candidates = []
    for pattern in ["checkpoints/*/pretrained_model", "checkpoint*/pretrained_model", "*/pretrained_model", "pretrained_model"]:
        candidates.extend(run_dir.glob(pattern))
    unique = sorted(set(candidates))
    if not unique:
        print("尚未发现 checkpoint：", public_path(run_dir))
        return []
    for path in unique:
        print(" -", public_path(path))
    return unique


def resolve_eval_policy(default_path, trained_run_dir=None, env_name=None):
    if env_name and os.environ.get(env_name):
        path = Path(os.environ[env_name])
        print("评估使用环境变量指定权重：", public_path(path))
        return path
    if trained_run_dir is not None and env_flag("EVAL_USE_LONG_TRAIN"):
        checkpoints = list_checkpoints(trained_run_dir)
        if checkpoints:
            path = checkpoints[-1]
            print("评估使用本次长训最新 checkpoint：", public_path(path))
            return path
        print("未找到本次长训 checkpoint，回退到保护权重。")
    path = Path(default_path)
    print("评估使用保护/预训练权重：", public_path(path))
    return path


def summarize_jsonl(path):
    path = Path(path)
    if not path.exists():
        print("结果 JSONL 尚不存在：", public_path(path))
        return None
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    total = len(rows)
    legacy = sum(bool(row.get("success") or row.get("legacy_success")) for row in rows)
    if rows and all("physical_success" in row for row in rows):
        physical_count = sum(bool(row.get("physical_success")) for row in rows)
        physical_text = str(physical_count) + "/" + str(total)
    else:
        physical_text = "未记录"
    md_table(
        ["结果文件", "episodes", "legacy_success", "physical_success"],
        [(public_path(path), total, f"{legacy}/{total}", physical_text)],
    )
    return rows


## Checkpoint 1：先确认这一版为什么作为主线


In [3]:
rows = [
    ("历史教程记录", "53/60", "SmolVLA weighted step500，红蓝杯平衡较好"),
    ("当前重建结果", "57/60", "red 27/30，blue 30/30"),
    ("发布建议", "主推权重", "适合作为零训练预览和默认 Notebook 案例"),
]
md_table(["项目", "结果", "说明"], rows)


| 项目 | 结果 | 说明 |
| --- | --- | --- |
| 历史教程记录 | 53/60 | SmolVLA weighted step500，红蓝杯平衡较好 |
| 当前重建结果 | 57/60 | red 27/30，blue 30/30 |
| 发布建议 | 主推权重 | 适合作为零训练预览和默认 Notebook 案例 |

## Checkpoint 2：显示严格成功与失败视频


## Checkpoint 1.5：保护权重的训练配方，不和课堂轻量训练混用

            上一个单元给出的是已经保护的 `57/60` 结果。这里说明它是怎么训练出来的。课堂默认长训只是让学习者体验完整流程；要复现保护权重，需要切到下面的 protected recipe，并跑完整训练与 60 episode strict eval。


### 结果口径对齐：本轮小面板与正式保护评估

            Notebook 后面的 eval 单元会跑一个便宜的小面板，用来确认本轮 checkpoint 能闭环执行并产出视频；正式保护评估使用更大的固定面板，才是发布权重和教程报告采用的口径。

            | 口径 | 成功率 | 评估范围 | 教学解释 |
            | --- | --- | --- | --- |
            | 本轮 Notebook 小面板 | `3/4` | post-long eval seed0-3 | 用于课堂展示和视频验收，不替代正式分数 |
            | 正式保护评估 | `57/60` | red30 + blue30 strict physical success | 红 `27/30`、蓝 `30/30`，作为默认发布权重 |


In [4]:
# PROTECTED_RECIPE_CELL
rows = [
    ("教学默认", "demo_data_language", "普通 EpisodeAwareSampler", "SMOLVLA_STEPS=5000", "本轮小面板 3/4", "用于课堂跑通和视频展示，不保证得到 57/60"),
    ("保护配方 parent", "demo_data_language", "基础 SmolVLA 长训", "5000 steps", "作为加权续训父权重", "先得到可用 parent checkpoint"),
    ("保护配方 weighted-blue", "demo_data_language", "蓝杯 frame/episode 加权，不复制原始 parquet", "续训 1000 steps，选择 step500", "red30 + blue30 strict", "当前重建 57/60；红 27/30，蓝 30/30"),
]
md_table(["模式", "数据", "采样/数据策略", "训练步数", "评估面板", "教学解释"], rows)

protected_env = {
    "TEACHING_RECIPE": "protected",
    "TRAIN_DATA_ROOT": str(DATA_ROOT / "demo_data_language"),
    "SMOLVLA_STEPS": "5000 + weighted-blue continuation",
    "SMOLVLA_EVAL_EPISODES": "60",
    "SMOLVLA_POLICY_PATH": str(MODEL_ROOT / "smolvla_weighted_000500" / "pretrained_model"),
}
print(json.dumps(protected_env, ensure_ascii=False, indent=2))
print("注意：weighted-blue 采样逻辑必须和 README_04/README_06 中的 Weighted sampler 一致；普通课堂长训不能冒充 protected 结果。")


{
  "TEACHING_RECIPE": "protected",
  "TRAIN_DATA_ROOT": "$TRAIN_DATA_ROOT",
  "SMOLVLA_STEPS": "5000 + weighted-blue continuation",
  "SMOLVLA_EVAL_EPISODES": "60",
  "SMOLVLA_POLICY_PATH": "$MODEL_ROOT/smolvla_weighted_000500/pretrained_model"
}
注意：weighted-blue 采样逻辑必须和 README_04/README_06 中的 Weighted sampler 一致；普通课堂长训不能冒充 protected 结果。


| 模式 | 数据 | 采样/数据策略 | 训练步数 | 评估面板 | 教学解释 |
| --- | --- | --- | --- | --- | --- |
| 教学默认 | demo_data_language | 普通 EpisodeAwareSampler | SMOLVLA_STEPS=5000 | 本轮小面板 3/4 | 用于课堂跑通和视频展示，不保证得到 57/60 |
| 保护配方 parent | demo_data_language | 基础 SmolVLA 长训 | 5000 steps | 作为加权续训父权重 | 先得到可用 parent checkpoint |
| 保护配方 weighted-blue | demo_data_language | 蓝杯 frame/episode 加权，不复制原始 parquet | 续训 1000 steps，选择 step500 | red30 + blue30 strict | 当前重建 57/60；红 27/30，蓝 30/30 |

### 可执行 protected 训练：parent 5000 + weighted-blue 续训

            设置 `RUN_PROTECTED_TRAIN=1` 后，这一格会在 Notebook 内部真实训练两个阶段：先训练 parent，再把 parent checkpoint 作为初始化继续做 blue 加权续训。默认关闭是为了避免课堂一打开就长训。


In [5]:
# PROTECTED_TRAIN_CELL
protected_train_enabled = env_flag("RUN_PROTECTED_TRAIN", False)
if not protected_train_enabled:
    print("未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 SmolVLA protected recipe。")
else:
    DATASET_REPO_ID = globals().get("DATASET_REPO_ID", os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language"))
    TRAIN_DATA_ROOT = globals().get("TRAIN_DATA_ROOT", Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "demo_data_language")))
    CONFIG_DIR = OUTPUT_ROOT / "configs"
    RUN_ROOT = OUTPUT_ROOT / "runs" / "smolvla_protected_recipe"
    PARENT_OUTPUT = RUN_ROOT / "parent_5000"
    WEIGHTED_OUTPUT = RUN_ROOT / "weighted_blue2_step1000"
    parent_config = make_lerobot_train_config(
        "smolvla", DATASET_REPO_ID, TRAIN_DATA_ROOT, PARENT_OUTPUT,
        steps=int(os.environ.get("SMOLVLA_PARENT_STEPS", "5000")),
        batch_size=int(os.environ.get("SMOLVLA_BATCH_SIZE", "4")),
        chunk_size=50,
        n_action_steps=50,
    )
    weighted_config = make_lerobot_train_config(
        "smolvla", DATASET_REPO_ID, TRAIN_DATA_ROOT, WEIGHTED_OUTPUT,
        steps=int(os.environ.get("SMOLVLA_WEIGHTED_STEPS", "1000")),
        batch_size=int(os.environ.get("SMOLVLA_BATCH_SIZE", "4")),
        chunk_size=50,
        n_action_steps=50,
    )
    parent_path = write_json_yaml(CONFIG_DIR / "smolvla_protected_parent_5000.yaml", parent_config)
    weighted_path = write_json_yaml(CONFIG_DIR / "smolvla_protected_weighted_blue2.yaml", weighted_config)
    train_lerobot_config_in_notebook(parent_path, enabled=True, progress_name="SmolVLA protected parent")
    parent_ckpt = list_checkpoints(PARENT_OUTPUT)[-1]
    old_override = os.environ.get("SMOLVLA_PRETRAINED_PATH_OVERRIDE")
    old_mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE")
    old_blue = os.environ.get("NOTEBOOK_BLUE_WEIGHT")
    os.environ["SMOLVLA_PRETRAINED_PATH_OVERRIDE"] = str(parent_ckpt)
    os.environ["NOTEBOOK_FRAME_WEIGHT_MODE"] = "blue"
    os.environ["NOTEBOOK_BLUE_WEIGHT"] = os.environ.get("SMOLVLA_BLUE_WEIGHT", "2.0")
    try:
        train_lerobot_config_in_notebook(weighted_path, enabled=True, progress_name="SmolVLA protected weighted-blue")
    finally:
        if old_override is None:
            os.environ.pop("SMOLVLA_PRETRAINED_PATH_OVERRIDE", None)
        else:
            os.environ["SMOLVLA_PRETRAINED_PATH_OVERRIDE"] = old_override
        if old_mode is None:
            os.environ.pop("NOTEBOOK_FRAME_WEIGHT_MODE", None)
        else:
            os.environ["NOTEBOOK_FRAME_WEIGHT_MODE"] = old_mode
        if old_blue is None:
            os.environ.pop("NOTEBOOK_BLUE_WEIGHT", None)
        else:
            os.environ["NOTEBOOK_BLUE_WEIGHT"] = old_blue
    print("protected candidate checkpoints:")
    list_checkpoints(WEIGHTED_OUTPUT)


未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 SmolVLA protected recipe。


In [6]:
show_video("smolvla_weighted500_red_success_seed0.mp4", "红杯成功回放：weighted500 seed0")
show_video("smolvla_weighted500_blue_success_seed0.mp4", "蓝杯成功回放：weighted500 seed0")
show_video("smolvla_weighted500_red_failure_seed8.mp4", "红杯失败回放：用于观察 upright/release 问题")


**红杯成功回放：weighted500 seed0**

**蓝杯成功回放：weighted500 seed0**

**红杯失败回放：用于观察 upright/release 问题**

## Checkpoint 3：检查数据和权重路径

            预计耗时：几秒。这里不会训练，只确认数据、权重和输出目录是否指向正确位置。


In [7]:
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language")
TRAIN_DATA_ROOT = Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "demo_data_language"))
PRETRAINED_POLICY = Path(os.environ.get("SMOLVLA_POLICY_PATH", MODEL_ROOT / "smolvla_weighted_000500" / "pretrained_model"))

rows = [
    ("DATASET_REPO_ID", DATASET_REPO_ID),
    ("TRAIN_DATA_ROOT", TRAIN_DATA_ROOT),
    ("SMOLVLA_POLICY_PATH", PRETRAINED_POLICY),
    ("数据 meta", TRAIN_DATA_ROOT / "meta" / "info.json"),
]
md_table(["变量", "当前值"], rows)


| 变量 | 当前值 |
| --- | --- |
| DATASET_REPO_ID | datawhale_eai_pnp_language |
| TRAIN_DATA_ROOT | $DATA_ROOT/demo_data_language |
| SMOLVLA_POLICY_PATH | $OUTPUT_ROOT/runs/smolvla_protected_recipe/weighted_blue2_step1000/checkpoints/001000/pretrained_model |
| 数据 meta | $DATA_ROOT/demo_data_language/meta/info.json |

## Checkpoint 4：生成配置并真实启动 smoke / 长训

            预计耗时：smoke 约 1-5 分钟；`SMOLVLA_STEPS=5000` 的长训通常需要几十分钟到数小时，取决于 ROCm、batch size 和数据盘速度。  
            这一格是真实训练入口：设置 `RUN_SMOKE=1` 或 `RUN_LONG_TRAIN=1` 后执行，会在 Notebook kernel 内直接创建 dataset、policy、optimizer 和训练循环，并显示 tqdm 进度。


In [8]:
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RUN_ROOT = OUTPUT_ROOT / "runs" / "smolvla_weighted_repro"
SMOKE_OUTPUT = RUN_ROOT / "smoke"
LONG_OUTPUT = RUN_ROOT / "weighted_full"

smoke_config = make_lerobot_train_config(
    "smolvla", DATASET_REPO_ID, TRAIN_DATA_ROOT, SMOKE_OUTPUT,
    steps=2, batch_size=2, chunk_size=50, n_action_steps=50,
)
long_config = make_lerobot_train_config(
    "smolvla", DATASET_REPO_ID, TRAIN_DATA_ROOT, LONG_OUTPUT,
    steps=int(os.environ.get("SMOLVLA_STEPS", "5000")),
    batch_size=int(os.environ.get("SMOLVLA_BATCH_SIZE", "4")),
    chunk_size=50,
    n_action_steps=50,
)
smoke_config_path = write_json_yaml(CONFIG_DIR / "smolvla_smoke.yaml", smoke_config)
long_config_path = write_json_yaml(CONFIG_DIR / "smolvla_weighted_full.yaml", long_config)

train_lerobot_config_in_notebook(smoke_config_path, enabled=RUN_SMOKE, progress_name="SmolVLA smoke")
train_lerobot_config_in_notebook(long_config_path, enabled=RUN_LONG_TRAIN, progress_name="SmolVLA long train")


写出配置： $OUTPUT_ROOT/configs/smolvla_smoke.yaml
写出配置： $OUTPUT_ROOT/configs/smolvla_weighted_full.yaml
config = $OUTPUT_ROOT/configs/smolvla_smoke.yaml
未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。
config = $OUTPUT_ROOT/configs/smolvla_weighted_full.yaml
未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。


## Checkpoint 5：实时查看训练日志和 checkpoint

            预计耗时：几秒。长训进行中可以反复执行本格，查看 Notebook 训练写出的 metrics 和已经落盘的 checkpoint。


In [9]:
print("smoke metrics:")
tail_log(SMOKE_OUTPUT / "notebook_train_metrics.jsonl", lines=20)
print("\nlong train metrics:")
tail_log(LONG_OUTPUT / "notebook_train_metrics.jsonl", lines=40)
print("\ncheckpoints:")
list_checkpoints(LONG_OUTPUT)


smoke metrics:
日志不存在： $OUTPUT_ROOT/runs/smolvla_weighted_repro/smoke/notebook_train_metrics.jsonl

long train metrics:
日志不存在： $OUTPUT_ROOT/runs/smolvla_weighted_repro/weighted_full/notebook_train_metrics.jsonl

checkpoints:
尚未发现 checkpoint： $OUTPUT_ROOT/runs/smolvla_weighted_repro/weighted_full


## Checkpoint 6：已完成长训的实测对照

            这是我们已经在 AMD 设备上跑完并复核过的视频/指标对照，用来帮助学习者先看到“正确跑起来是什么样”。它不替代本次 Notebook 的训练输出。


In [10]:
rows = [
    ("parent", "5000/5000 steps", "基础 SmolVLA 收敛到可用 checkpoint"),
    ("weighted-blue", "selected step 500/1000", "step500 比 step1000 更平衡"),
    ("strict gate", "57/60", "legacy 60/60，physical 57/60"),
]
md_table(["阶段", "训练/评估进度", "结论"], rows)


| 阶段 | 训练/评估进度 | 结论 |
| --- | --- | --- |
| parent | 5000/5000 steps | 基础 SmolVLA 收敛到可用 checkpoint |
| weighted-blue | selected step 500/1000 | step500 比 step1000 更平衡 |
| strict gate | 57/60 | legacy 60/60，physical 57/60 |

In [11]:
show_image("training_progress_overview.png", "历史训练进度与闭环结果")
show_image("smolvla_red_blue_success.png", "红杯/蓝杯分指令对比")


**历史训练进度与闭环结果**

**红杯/蓝杯分指令对比**

## Checkpoint 7：Notebook 内严格评估

            预计耗时：10 个 episode 通常 10-30 分钟；`SMOLVLA_EVAL_EPISODES=60` 会更久。  
            本单元会在 Notebook kernel 内加载策略并逐个 seed 闭环 rollout；如无显示器，会自动尝试启动 Xvfb。


In [12]:
eval_episodes = os.environ.get("SMOLVLA_EVAL_EPISODES", "10")
eval_policy = resolve_eval_policy(PRETRAINED_POLICY, LONG_OUTPUT, "SMOLVLA_EVAL_POLICY_PATH")
run_eval_policy_in_notebook(
    "smolvla",
    eval_policy,
    OUTPUT_ROOT / f"smolvla_eval_seed0_{int(eval_episodes)-1}.jsonl",
    episodes=eval_episodes,
    seed_start=0,
    render=env_flag("RENDER_EVAL"),
    enabled=RUN_EVAL,
    repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
)
summarize_jsonl(OUTPUT_ROOT / f"smolvla_eval_seed0_{int(eval_episodes)-1}.jsonl")


评估使用环境变量指定权重： $OUTPUT_ROOT/runs/smolvla_protected_recipe/weighted_blue2_step1000/checkpoints/001000/pretrained_model
policy = smolvla
policy_path = $OUTPUT_ROOT/runs/smolvla_protected_recipe/weighted_blue2_step1000/checkpoints/001000/pretrained_model
result = $OUTPUT_ROOT/smolvla_eval_seed0_59.jsonl
没有发现 Xvfb；如遇 GLFW DISPLAY 报错，请先安装 xvfb。


FileNotFoundError: 评估脚本不存在：$PROJECT_ROOT/eval_policy_success.py

## Checkpoint 8：怎么读结果


In [13]:
summary = {
    "candidate": "weighted_000500",
    "episodes": 60,
    "physical_success_count": 57,
    "legacy_success_count": 60,
    "by_color": {"red": "27/30", "blue": "30/30"},
    "release_decision": "作为教程默认预训练权重",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "candidate": "weighted_000500",
  "episodes": 60,
  "physical_success_count": 57,
  "legacy_success_count": 60,
  "by_color": {
    "red": "27/30",
    "blue": "30/30"
  },
  "release_decision": "作为教程默认预训练权重"
}
